<a href="https://colab.research.google.com/github/hasbiazif/Belajar/blob/Belajar/Contoh_scraping_web.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cara Scraping we menggunakan BeautifulSoup**

## Ambil setiap link dari halamannya

In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://www.hukumonline.com/klinik/pidana/page/"
max_pages = 1  # Ubah jumlah halaman maksimum sesuai kebutuhan

all_klinik_a_links = [] # list untuk menyimpan semua link dari semua page

for page_number in range(1, max_pages + 1):
    url = base_url + str(page_number) + "/"
    print(f"Mengambil data dari halaman: {url}")

    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        target_div = soup.find("div", class_="css-1y720r3")

        if target_div:
            a_elements = target_div.find_all("a")
            for a in a_elements:
                href = a["href"]
                if "/klinik/a/" in href:
                    full_link = "https://www.hukumonline.com" + href # Tambahkan domain
                    all_klinik_a_links.append(full_link)
                    # print(f"Link: {full_link}")

        else:
            print(f"Div dengan class 'css-1y720r3' tidak ditemukan di halaman {page_number}.")

    except requests.exceptions.RequestException as e:
        print(f"Terjadi kesalahan saat mengambil URL {url}: {e}")
    except Exception as e:
        print(f"Terjadi kesalahan di halaman {page_number}: {e}")

print("\nSemua link /klinik/a/ dari semua halaman:", all_klinik_a_links)

## mengambil data dari link yang diambil

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.hukumonline.com/klinik/a/dasar-hukum-penenggelaman-kapal-asing-pencuri-ikan-lt54e31f284a8ff/"

try:
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    target_div = soup.find("div", class_="css-33k52x")

    if target_div:
        # Mendapatkan semua teks di dalam div, dengan mempertahankan struktur baris
        all_text = target_div.get_text(separator='\n', strip=True)
        print(all_text)

    else:
        print("Div dengan class 'css-33k52x' tidak ditemukan.")

except requests.exceptions.RequestException as e:
    print(f"Terjadi kesalahan saat mengambil URL: {e}")
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

# **Fungsi Lengkap**

*   Mengambil link
*   Mengambil data dari setiap link
*   fungsi embed
*   fungsi upload ke qdrant





In [ ]:
!pip install qdrant_client

In [ ]:
import json
import re
import requests
from bs4 import BeautifulSoup
from qdrant_client import QdrantClient, models

def extract_title_from_url(url):
    try:
        modified_url = url.removeprefix("https://www.hukumonline.com/klinik/a/")
        title = re.sub(r'-[a-z0-9]+/$', '', modified_url)
        return title.strip()
    except Exception as e:
        print(f"Error extracting title from URL '{url}': {e}")
        return None

def process_article(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        target_div = soup.find("div", class_="css-33k52x")
        text = ""
        if target_div:
            text = target_div.get_text(separator='\n', strip=True)
            text = text.replace("Beranda\nKlinik\nPidana\nPidana\n", "")
            text = text.replace("\n", " ")

        judul = extract_title_from_url(url)
        if judul is None:
            return None

        text_lower = text.lower()

        pattern_pertanyaan = r"(?s)pertanyaan(.*?)(?=intisari jawaban)"
        pattern_intisari = r"(?s)intisari jawaban(.*?)(?=ulasan lengkap)"
        pattern_ulasan = r"(?s)ulasan lengkap(.*?)(?=dasar hukum)"
        pattern_dasar = r"(?s)dasar hukum(.*)"

        match_pertanyaan = re.search(pattern_pertanyaan, text_lower)
        match_intisari = re.search(pattern_intisari, text_lower)
        match_ulasan = re.search(pattern_ulasan, text_lower)
        match_dasar = re.search(pattern_dasar, text_lower)

        result_item = {
            "url": url,
            "judul": judul,
            "pertanyaan": match_pertanyaan.group(1).strip() if match_pertanyaan else "",
            "intisari_jawaban": match_intisari.group(1).strip() if match_intisari else "",
            "ulasan_lengkap": match_ulasan.group(1).strip() if match_ulasan else "",
            "dasar_hukum": match_dasar.group(1).strip() if match_dasar else ""
        }
        return result_item
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL {url}: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while processing {url}: {e}")
        return None

def scrape_page(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        target_div = soup.find("div", class_="css-1y720r3")
        links = []
        if target_div:
            a_elements = target_div.find_all("a")
            for a in a_elements:
                href = a["href"]
                if "/klinik/a/" in href:
                    full_link = "https://www.hukumonline.com" + href
                    links.append(full_link)
        return links
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL {url}: {e}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred while processing {url}: {e}")
        return []

def embed_user_query(user_queries: list):
    url = "http://207.148.119.33:42003/embeddings"
    headers = {"Content-Type": "application/json"}
    payload = {"input": user_queries}

    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
        return response.json()["embeddings"]
    except requests.exceptions.RequestException as e:
        print(f"Error fetching embeddings: {e}")
        return [None] * len(user_queries)
    except (KeyError, IndexError) as e:
        print(f"Error processing embedding response: {e}")
        return [None] * len(user_queries)
    except Exception as e:
        print(f"An unexpected error occurred while getting embeddings: {e}")
        return [None] * len(user_queries)


def upload_to_qdrant(articles_data, embeddings, qdrant_client, collection_name):
    payloads = [
        {
            "id": i,
            "vector": embedding,
            "payload": article,
        }
        for i, (article, embedding) in enumerate(zip(articles_data, embeddings))
        if article is not None and embedding is not None
    ]

    try:
        qdrant_client.upsert(collection_name=collection_name, points=payloads)
        print(f"Successfully uploaded {len(payloads)} points to Qdrant.")
    except Exception as e:
        print(f"Error uploading to Qdrant: {e}")

def save_data_to_json(data, filename="scraped_data.json"):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_data_from_json(filename="scraped_data.json"):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in '{filename}'.")
        return []

## proses menambil data dari setiap link

In [ ]:
base_url = "https://www.hukumonline.com/klinik/pidana/page/"
max_pages = 2
all_data = []

for page_number in range(1, max_pages + 1):
    url = base_url + str(page_number) + "/"
    print(f"Processing page: {url}")
    links = scrape_page(url)
    for link in links:
        result = process_article(link)
        if result:
            all_data.append(result)

save_data_to_json(all_data)

## proses upload ke qdrant

In [ ]:
qdrant = QdrantClient(host="46.250.225.100", port=36333)
collection_name = "Hasbi_scr"
vector_size = 768
embedding_url = "http://207.148.119.33:42003/embeddings"

articles_data = load_data_from_json()
if articles_data:
    texts_to_embed = [
        article["pertanyaan"] + " " + article["intisari_jawaban"] + " " + article["ulasan_lengkap"] + " " + article["dasar_hukum"]
        for article in articles_data
    ]
    try:
        embeddings = embed_user_query(texts_to_embed)
        upload_to_qdrant(articles_data, embeddings, qdrant, collection_name)
    except Exception as e:
        print(f"Error during embedding or upload: {e}")